<a href="https://colab.research.google.com/github/bhavana-2005-stack/Mini_LLM_Model/blob/main/Mini_LLM_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kagglehub

In [2]:
!pip install torch pandas numpy tqdm -q

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hgultekin/bbcnewsarchive")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'bbcnewsarchive' dataset.
Path to dataset files: /kaggle/input/bbcnewsarchive


In [4]:
import os
print(os.listdir(path))

['bbc-news-data.csv']


In [5]:
import pandas as pd
import math
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
print("PyTorch:", torch.__version__)

PyTorch: 2.11.0+cu128


In [6]:
for root, dirs, files in os.walk(path):
    for file in files:
        print(file)

bbc-news-data.csv


In [7]:
csv_file = os.path.join(path, "bbc-news-data.csv")

df = pd.read_csv(
    csv_file,
    sep="\t"
)

print(df.head())
print(df.shape)

   category filename                              title  \
0  business  001.txt  Ad sales boost Time Warner profit   
1  business  002.txt   Dollar gains on Greenspan speech   
2  business  003.txt  Yukos unit buyer faces loan claim   
3  business  004.txt  High fuel prices hit BA's profits   
4  business  005.txt  Pernod takeover talk lifts Domecq   

                                             content  
0   Quarterly profits at US media giant TimeWarne...  
1   The dollar has hit its highest level against ...  
2   The owners of embattled Russian oil giant Yuk...  
3   British Airways has blamed high fuel prices f...  
4   Shares in UK drinks and food firm Allied Dome...  
(2225, 4)


In [8]:
df.columns

Index(['category', 'filename', 'title', 'content'], dtype='object')

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   category  2225 non-null   object
 1   filename  2225 non-null   object
 2   title     2225 non-null   object
 3   content   2225 non-null   object
dtypes: object(4)
memory usage: 69.7+ KB


In [10]:
df.describe()

,category,filename,title,content
count,2225,2225,2225,2225
unique,5,511,2096,2092
top,sport,385.txt,Doors open at biggest gadget fair,Fast web access is encouraging more people to...
freq,511,5,2,2


In [11]:
print("Total Articles:", len(df))

Total Articles: 2225


In [12]:
texts = df["content"].astype(str).tolist()

print("Number of Articles:", len(texts))
print(texts[0][:500])

Number of Articles: 2225
 Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.  Time Warner said on Friday that it now o


In [13]:
full_text = " ".join(texts)

print("Characters:", len(full_text))

Characters: 4970189


In [14]:
tokens = full_text.lower().split()

print("Total Tokens:", len(tokens))
print(tokens[:20])

Total Tokens: 842910
['quarterly', 'profits', 'at', 'us', 'media', 'giant', 'timewarner', 'jumped', '76%', 'to', '$1.13bn', '(£600m)', 'for', 'the', 'three', 'months', 'to', 'december,', 'from', '$639m']


In [15]:
counter = Counter(tokens)

vocab = ["<PAD>", "<UNK>"] + sorted(counter.keys())

word_to_idx = {
    word:i
    for i, word in enumerate(vocab)
}

idx_to_word = {
    i:word
    for word,i in word_to_idx.items()
}

vocab_size = len(vocab)

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 59969


In [16]:
encoded = [
    word_to_idx.get(
        word,
        word_to_idx["<UNK>"]
    )
    for word in tokens
]

print(encoded[:20])

[44362, 43676, 10755, 56458, 36344, 26939, 54370, 32520, 7519, 54451, 2462, 4692, 25534, 53856, 54134, 37532, 54451, 19471, 26079, 2990]


In [17]:
SEQ_LEN = 20

inputs = []
targets = []

for i in range(len(encoded) - SEQ_LEN):

    x = encoded[i:i+SEQ_LEN]

    y = encoded[i+1:i+SEQ_LEN+1]

    inputs.append(x)
    targets.append(y)

print(len(inputs))

842890


In [18]:
MAX_SAMPLES = 30000

inputs = inputs[:MAX_SAMPLES]
targets = targets[:MAX_SAMPLES]

print(len(inputs))

30000


In [19]:
class GPTDataset(Dataset):

    def __init__(self, x, y):

        self.x = torch.tensor(x)
        self.y = torch.tensor(y)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [20]:
dataset = GPTDataset(
    inputs,
    targets
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

In [21]:
class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=5000
    ):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ) * (-math.log(10000.0) / d_model)
        )

        pe[:,0::2] = torch.sin(
            position * div_term
        )

        pe[:,1::2] = torch.cos(
            position * div_term
        )

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):

        return x + self.pe[:, :x.size(1)]

In [22]:
class MaskedMultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads
    ):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, x):

        seq_len = x.size(1)

        mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len
            ),
            diagonal=1
        ).bool().to(x.device)

        out, _ = self.attn(
            x,
            x,
            x,
            attn_mask=mask
        )

        return out

In [23]:
class MaskedMultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads
    ):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, x):

        seq_len = x.size(1)

        mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len
            ),
            diagonal=1
        ).bool().to(x.device)

        out, _ = self.attn(
            x,
            x,
            x,
            attn_mask=mask
        )

        return out

In [24]:
class GPTBlock(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        ff_dim
    ):
        super().__init__()

        self.attn = MaskedMultiHeadAttention(
            d_model,
            num_heads
        )

        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        x = self.norm1(
            x + self.attn(x)
        )

        x = self.norm2(
            x + self.ff(x)
        )

        return x

In [25]:
class MiniGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=128,
        num_heads=4,
        ff_dim=256,
        num_layers=4,
        max_len=100
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_encoding = PositionalEncoding(
            d_model,
            max_len
        )

        self.blocks = nn.ModuleList(
            [
                GPTBlock(
                    d_model,
                    num_heads,
                    ff_dim
                )
                for _ in range(num_layers)
            ]
        )

        self.fc = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, x):

        x = self.embedding(x)

        x = self.pos_encoding(x)

        for block in self.blocks:
            x = block(x)

        return self.fc(x)

In [26]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = MiniGPT(
    vocab_size=vocab_size
).to(device)

print(model)

MiniGPT(
  (embedding): Embedding(59969, 128)
  (pos_encoding): PositionalEncoding()
  (blocks): ModuleList(
    (0-3): 4 x GPTBlock(
      (attn): MaskedMultiHeadAttention(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
        (0): Linear(in_features=128, out_features=256, bias=True)
        (1): ReLU()
        (2): Linear(in_features=256, out_features=128, bias=True)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
  )
  (fc): Linear(in_features=128, out_features=59969, bias=True)
)


In [27]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [28]:
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for x, y in tqdm(loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(
            outputs.reshape(-1, vocab_size),
            y.reshape(-1)
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
        f" Loss={total_loss/len(loader):.4f}"
    )

100%|██████████| 938/938 [00:22<00:00, 41.27it/s]


Epoch 1/5 Loss=4.4318


100%|██████████| 938/938 [00:22<00:00, 42.38it/s]


Epoch 2/5 Loss=0.6855


100%|██████████| 938/938 [00:22<00:00, 41.62it/s]


Epoch 3/5 Loss=0.4422


100%|██████████| 938/938 [00:23<00:00, 40.26it/s]


Epoch 4/5 Loss=0.3916


100%|██████████| 938/938 [00:23<00:00, 40.17it/s]

Epoch 5/5 Loss=0.3666


In [29]:
def generate_text(
    prompt,
    max_new_tokens=20
):

    model.eval()

    words = prompt.lower().split()

    ids = [
        word_to_idx.get(
            w,
            word_to_idx["<UNK>"]
        )
        for w in words
    ]

    for _ in range(max_new_tokens):

        inp = torch.tensor(
            [ids[-SEQ_LEN:]],
            device=device
        )

        with torch.no_grad():
            out = model(inp)

        next_token = torch.argmax(
            out[0,-1]
        ).item()

        ids.append(next_token)

    return " ".join(
        idx_to_word[i]
        for i in ids
    )

In [30]:
print(
    generate_text(
        "artificial intelligence",
        max_new_tokens=30
    )
)

artificial intelligence including daimlerchrysler, siemens and volkswagen, have been negotiating with unions over cost cutting measures. analysts said that the ifo figures and germany's continuing problems may delay an interest rate rise


In [31]:
print(
    generate_text(
        "Politics",
        max_new_tokens=30
    )
)

politics foreign investors has been hurt in recent months by the heavy tax bills and asset seizures imposed on companies such as oil giant yukos. s&p said the solidity of government


In [32]:
torch.save(model.state_dict(), "mini_gpt_model.pth")

In [33]:
import pickle

with open("word_to_idx.pkl", "wb") as f:
    pickle.dump(word_to_idx, f)

with open("idx_to_word.pkl", "wb") as f:
    pickle.dump(idx_to_word, f)